# 🔬 Model B: CNN with Ordinal Regression (Frank-Hall Decomposition)
### Core Academic Research Contribution
This notebook implements our main research model: **CNN + Ordinal Regression**.

### Mathematical Formulation:
Fruit ripeness follows a natural progression:
$$\text{Unripe (0)} < \text{Ripe (1)} < \text{Overripe (2)}$$

Instead of unordered softmax, we decompose the $K=3$ ranking problem into $K-1 = 2$ cumulative binary threshold questions:
1. **Head 1:** $P(Y \ge 1)$ — *Is the fruit at least ripe?*
2. **Head 2:** $P(Y \ge 2)$ — *Is the fruit overripe?*

| True Ripeness | Class Index $y$ | Head 1 Target: $Y \ge 1$ | Head 2 Target: $Y \ge 2$ |
|---|---|---|---|
| **Unripe** | 0 | 0 | 0 |
| **Ripe** | 1 | 1 | 0 |
| **Overripe** | 2 | 1 | 1 |


In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import numpy as np
from PIL import Image

PROJECT_ROOT = Path("..").resolve()
DATASET_DIR = PROJECT_ROOT / "dataset"
MODELS_DIR = PROJECT_ROOT / "models"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 15


## 1. Cumulative Ordinal Encoding Function


In [ ]:
def to_ordinal_targets(y):
    # Encodes integer ranks {0, 1, 2} into cumulative binary vectors [Y>=1, Y>=2]
    targets = np.zeros((len(y), 2), dtype=np.float32)
    targets[:, 0] = (y >= 1).astype(np.float32)
    targets[:, 1] = (y >= 2).astype(np.float32)
    return targets

def decode_ordinal(probs):
    p1, p2 = probs[:, 0], probs[:, 1]
    p1_mono = np.maximum(p1, p2)
    p2_mono = p2
    p_unripe = np.clip(1.0 - p1_mono, 0.0, 1.0)
    p_ripe = np.clip(p1_mono - p2_mono, 0.0, 1.0)
    p_overripe = np.clip(p2_mono, 0.0, 1.0)
    class_probs = np.stack([p_unripe, p_ripe, p_overripe], axis=-1)
    class_probs /= class_probs.sum(axis=1, keepdims=True)
    return np.argmax(class_probs, axis=1), class_probs


## 2. Ordinal Regression CNN Architecture


In [ ]:
inputs = layers.Input(shape=(224, 224, 3), name="input_image")
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.08)(x)

# Same backbone for rigorous scientific comparison
x = layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv3")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(256, (3, 3), padding="same", activation="relu", name="conv4_features")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dense(128, activation="relu", name="dense_shared")(x)
x = layers.Dropout(0.35)(x)

# 2 Sigmoid Threshold Outputs (Frank-Hall Formulation)
outputs = layers.Dense(2, activation="sigmoid", name="ordinal_thresholds")(x)

ord_model = keras.Model(inputs=inputs, outputs=outputs, name="Ordinal_CNN")
ord_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["binary_accuracy"]
)
ord_model.summary()


## 3. Training Ordinal Model with Cumulative BCE


In [ ]:
def load_split(split_name):
    split_dir = DATASET_DIR / split_name
    images, labels = [], []
    for c_idx, c_name in enumerate(["unripe", "ripe", "overripe"]):
        for img_p in (split_dir / c_name).glob("*.jpg"):
            img = Image.open(img_p).convert("RGB").resize(IMAGE_SIZE)
            images.append(np.array(img, dtype=np.float32) / 255.0)
            labels.append(c_idx)
    return np.array(images), np.array(labels)

X_train, y_train_raw = load_split("train")
X_val, y_val_raw = load_split("validation")

y_train_ord = to_ordinal_targets(y_train_raw)
y_val_ord = to_ordinal_targets(y_val_raw)

history_ord = ord_model.fit(
    X_train, y_train_ord,
    validation_data=(X_val, y_val_ord),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)
ord_model.save(MODELS_DIR / "ordinal_cnn.keras")
print("Saved Model B: ordinal_cnn.keras")
